# 1 Importing Packages

How the *MESSAGE_CA* class works: the class will contain all the functions necessary to run the MESSAGEix Canada model. All that you need to run a model is to:

1. Create your scenario folder under `model/scenarios` with its own `config.toml` file

2. Declare the class reading that scenario, for example: `platform, msg, msg_folder = MESSAGE_CA(scenario_name='baseline')`

In [1]:
import os
import sys


In [3]:
from model.Model import model_MSG 
from model.Plotter import plotter_MSG


# Autoreload modules when changiges are applied to them
%load_ext autoreload 
%autoreload all

In [2]:
model_MSG.clone_scenario

AttributeError: type object 'model_MSG' has no attribute 'clone_scenario'

In [2]:
# Sys and OS Operations

os.chdir("../../")

# Scenario Creation and Set-up

Setting up the MESSAGEix scenario using the scenario configuration file and an input file for technology options in technology.yaml

In [4]:
# Set-up
msg = model_MSG(scenario_name='baseline')
platform, msg_scenario, msg_folder = msg.setup_scenario()

Config file read successfully!
####################################################################################################
Config file 'model/scenarios\baseline\config.toml' passed format verification!
####################################################################################################

		Model creation time: 3 Min and 39 Sec 		


In [5]:
import pandas as pd
import pyam as pyam
import message_ix as message_ix

In [9]:
msg_scenario.remove_solution()

In [10]:
msg_scenario.check_out()

In [7]:
import pandas as pd

# Define the provinces and the years
provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
             'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
             'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

years = list(range(2030, 2061, 5))  # 2030, ..., 2060
technology = 'h2_atr_ccs'
value = 0.015

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        msg_scenario.add_par('initial_activity_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
        
msg_scenario.add_par('initial_activity_up', 
            [province, technology, "2020", "year"], 
            0.006, 'GWa')

msg_scenario.add_par('initial_activity_up', 
            [province, technology, "2025", "year"], 
            0.0105, 'GWa')

In [8]:
value = 0.08

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        msg_scenario.add_par('growth_activity_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

In [9]:
value = 0.006
technologies= ['h2_atr','h2_atr_ccs'] #meed to do for both w and without ccs

# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('initial_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

In [10]:
value = 0.05 #meed to do for both w and without ccs
technologies= ['h2_atr','h2_atr_ccs']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('growth_new_capacity_up', 
                    [province, technology, year], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed

In [11]:
value = 0.5 #meed to do for both w and without ccs
technologies= ['h2_atr','h2_atr_ccs']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('level_cost_activity_soft_lo', 
                    [province, technology, year, "year"], 
                    value, 'GWa')  # Assuming 'GWa' is the unit, adjust if needed
            msg_scenario.add_par('level_cost_activity_soft_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa')

In [12]:
value = -0.05 #meed to do for both w and without ccs
technologies= ['h2_atr','h2_atr_ccs']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('soft_activity_lo', 
                    [province, technology, year, "year"], 
                    value, 'GWa') 
            
value = 0.05 #meed to do for both w and without ccs
technologies= ['h2_atr','h2_atr_ccs']
# Loop through provinces and years to add the 'initial_activity_up' parameter
for province in provinces:
    for year in years:
        for technology in technologies:
            msg_scenario.add_par('soft_activity_up', 
                    [province, technology, year, "year"], 
                    value, 'GWa') 

In [13]:
msg_scenario.commit("added initial and growth up parameters for h2_atr_ccs")

In [14]:
msg_scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"},
				)

In [13]:

"""
Apply the Output-Based Pricing System (OBPS) policy for the Baseline Canada Scenario
to the given MESSAGEix scenario.

This function modifies the scenario's fixed costs based on emissions intensity and carbon pricing.

Args:
    scenario (ixmp.Scenario): The MESSAGEix scenario to apply the OBPS policy to.
    data_file_path (str): Path to the Excel file containing necessary data.
"""

# Read the required data
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')

provinces = ['Ontario', 'Quebec', 'BritishColumbia', 'Alberta', 'Manitoba', 
            'Saskatchewan', 'NovaScotia', 'NewBrunswick', 'NewfoundlandandLabrador', 
            'PrinceEdwardIsland', 'NorthwestTerritories', 'Yukon', 'Nunavut']

# Define conditions for filtering the data
conditions_2 = (
    data_relation['technology'].isin([
        'oil_extr_1', 'oil_extr_2', 'oil_extr_3', 'oil_extr_4', 'oil_extr_5', 'oil_extr_6',
        'cement_CO2', 'coal_extr', 'gas_extr_1', 'gas_extr_2', 'gas_extr_3', 'gas_extr_4', 
        'gas_extr_5', 'gas_extr_6', 'h2_smr_ccs', 'h2_smr', 'h2_atr', 'h2_atr_ccs'
    ]) &
    (data_relation['emission'] == 'CO2_cc') &
    (data_relation['node_loc'].isin(provinces))
)

# Filter the data based on the conditions
result = data_relation[conditions_2][['technology', 'value', 'node_loc', 'year_act']]

# Initialize columns for calculations
result['Emission Intensity Reduction_baseline'] = 0.0

# Calculate Emission Intensity Reduction_baseline
for index, row in result.iterrows():
    if 2020 < row['year_act'] < 2055:
        previous_year_value = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == row['year_act'] - 5), 
            'Emission Intensity Reduction_baseline'
        ].values
        if len(previous_year_value) > 0:
            result.loc[index, 'Emission Intensity Reduction_baseline'] += previous_year_value[0] + 3.33/100

# Calculate Emission intensity standard_Baseline Canada Scenario
result['Emission intensity standard_Baseline Canada Scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        value_cal = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2015), 
            'value'
        ]
        if not value_cal.empty:
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]
        else:
            value_cal = result.loc[
                (result['technology'] == row['technology']) & 
                (result['year_act'] == 2020), 
                'value'
            ]
            result.at[index, 'Emission intensity standard_Baseline Canada Scenario'] = row['value'] - row['Emission Intensity Reduction_baseline'] * value_cal.values[0]

# Calculate Payable emission intensity_Baseline scenario
result['Payable emission intensity_Baseline scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] > 2020:
        result.loc[index, 'Payable emission intensity_Baseline scenario'] = row['value'] - row['Emission intensity standard_Baseline Canada Scenario']

# Calculate carbon cost_Baseline Canada scenario
rate = 0.73
result['carbon cost_Baseline Canada scenario'] = 0.0
for index, row in result.iterrows():
    if row['year_act'] == 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 125
    elif row['year_act'] > 2025:
        result.loc[index, 'carbon cost_Baseline Canada scenario'] = row['Payable emission intensity_Baseline scenario'] * rate * 170

# Adjust for years 2055 and 2060
for index, row in result.iterrows():
    if row['year_act'] > 2050:
        value_to_assign = result.loc[
            (result['technology'] == row['technology']) & 
            (result['year_act'] == 2050), 
            'carbon cost_Baseline Canada scenario'
        ].values
        if len(value_to_assign) > 0:
            result.at[index, 'carbon cost_Baseline Canada scenario'] = value_to_assign[0]

# Merge with fix_cost data and update the scenario's fix_cost
data_relation = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='emission_factor')
# Read the fix_cost data
data_fixcost = pd.read_excel("model/scenarios/baseline/MESSAGE_Updated_LR.xlsx", sheet_name='fix_cost')

merged_data = pd.merge(
    data_fixcost, 
    result[['technology', 'year_act', 'node_loc', 'carbon cost_Baseline Canada scenario']],
    left_on=['technology', 'year_act', 'node_loc'],
    right_on=['technology', 'year_act', 'node_loc'],
    how='left'
)
merged_data['value'] = merged_data['value'] + merged_data['carbon cost_Baseline Canada scenario'].fillna(0)
merged_data.drop(columns=['carbon cost_Baseline Canada scenario'], inplace=True)

# Assuming 'merged_data' contains the data you've shown

for index, row in merged_data.iterrows():
    msg_scenario.add_par('fix_cost', pd.DataFrame({
        'node_loc': [row['node_loc']],
        'technology': [row['technology']],
        'year_vtg': [row['year_vtg']],
        'year_act': [row['year_act']],
        'mode': ['M1'],  # Assuming 'standard' mode for all entries
        'unit': [row['unit']],
        'value': [row['value']]
    }, index=[0]))

In [15]:
msg_scenario.to_excel(r"C:\Users\brend\Documents\GitHub\message-ix - CA\model\notebooks\MESSAGE_Updated_LR.xlsx")


In [16]:
output_path="C:/Users/brend/Documents/Github/message-ix - CA/scenarios/baseline/MESSAGE_test_updated_LR.csv"
min_year= 2015
max_year= 2060
from_excel=False
unit_correct=True

rep = message_ix.Reporter.from_scenario(
    msg.scenario,
    units={"apply": {"var_cost": "", "inv_cost": "", "fix_cost": ""}},
)
report = rep.get("message::default")

if from_excel:
    report_df = pd.read_csv("MESSAGE_NO_example_default_report.csv")
else:
    report_df = report.timeseries()
    report_df.reset_index(inplace=True)
    report_df.columns = report_df.columns.astype(str)
    report_df.columns = report_df.columns.str.title()

years = msg.scenario.set("year").tolist()
years = [str(x) for x in years if x >= min_year and x <= max_year]

report_df.fillna(0, inplace=True)

report_df.columns = report_df.columns.astype(str)
report_df = report_df.drop(
    report_df.columns.to_series()["2020" : str(min_year)], axis=1
)
report_df.to_csv('test.csv')
in_flows = report_df[report_df["Variable"].str.startswith("in|")]
out_flows = report_df[report_df["Variable"].str.startswith("out|")]
all_flows = pd.concat([out_flows, in_flows])

report_df["Model"] = "MESSAGEix-Canada"
report_df["Scenario"] = "Reference"
report_df["Variable"] = report_df["Variable"].str.replace(
    "emis", "Emissions", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|final|", "Final Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|primary|", "Primary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|secondary|", "Secondary Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|renewable|", "Primary Energy|renewable|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "total om cost|", "Total Costs|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "out|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "in|useful|", "Useful Energy|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP_NEW|new capacity|", "Capacity Addition|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "CAP|capacity|", "Capacity|", regex=False
)
report_df["Variable"] = report_df["Variable"].str.replace(
    "inv cost|", "Capital Cost|", regex=False
)
report_df["Region"] = report_df["Region"].apply(lambda x: x.split("|")[0])

report_df.columns = report_df.columns.str.strip()  # Remove any leading/trailing whitespace
report_df.rename(columns={'Model': 'model', 'Scenario': 'scenario', 'Variable': 'variable'}, inplace=True)

report_df = report_df.drop_duplicates(subset=["model", "scenario", "Region", "variable"], keep='first')


pyam_df = pyam.IamDataFrame(report_df)

variable = pyam_df.variable
pyam_df.aggregate_region(variable, append=True)
pyam_df.rename({"region": {"World": "Canada"}}, inplace=True)

pyam_df.to_csv(output_path)

print(f"Report saved to {output_path}")



var_cost: mixed units ['USD/kW', 'USD/kWa'] discarded
input: mixed units ['GWa', '-', '%'] discarded
emission_factor: mixed units ['tC', '-', 'GWa/a'] discarded
output: mixed units ['GWa', '-'] discarded
fix_cost: mixed units ['USD/kW', 'USD/kWa'] discarded
inv_cost: mixed units ['USD/kW', 'USD/kWa'] discarded


Report saved to C:/Users/brend/Documents/Github/message-ix - CA/scenarios/baseline/MESSAGE_test_updated_LR.csv


In [69]:
import os

output_dir = r'C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline'

# Check if the directory exists, if not, create it
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = os.path.join(output_dir, 'report.csv')

# Now save the file
pyam_df.to_csv(output_path)
print(f"Report saved to {output_path}")

Report saved to C:\Users\brend\Documents\Github\message-ix - CA\scenarios\baseline\report.csv


In [52]:
print(report_df["Region"].value_counts())

Region
Alberta                    3355
BritishColumbia            3355
Manitoba                   3355
NewBrunswick               3355
NewfoundlandandLabrador    3355
NorthwestTerritories       3355
NovaScotia                 3355
Nunavut                    3355
Ontario                    3355
PrinceEdwardIsland         3355
Quebec                     3355
Saskatchewan               3355
Yukon                      3355
Name: count, dtype: int64


In [59]:
report_df[report_df["variable"] =='Secondary Energy|electr|h2_smr_ccs|M1']


,model,scenario,Region,variable,Unit,2010,2015,2020,2025,2030,2035,2040,2045,2050,2055,2060
2816,MESSAGEix-Canada,Reference,Alberta,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3194,MESSAGEix-Canada,Reference,Alberta,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6171,MESSAGEix-Canada,Reference,BritishColumbia,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6549,MESSAGEix-Canada,Reference,BritishColumbia,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9526,MESSAGEix-Canada,Reference,Manitoba,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9904,MESSAGEix-Canada,Reference,Manitoba,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12881,MESSAGEix-Canada,Reference,NewBrunswick,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
13259,MESSAGEix-Canada,Reference,NewBrunswick,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16236,MESSAGEix-Canada,Reference,NewfoundlandandLabrador,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
16614,MESSAGEix-Canada,Reference,NewfoundlandandLabrador,Secondary Energy|electr|h2_smr_ccs|M1,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
print(report_df["variable"].value_counts())

variable
Secondary Energy|electr|h2_smr_ccs|M1                             26
Secondary Energy|electr|h2_coal|M1                                26
Secondary Energy|electr|h2_bio|M1                                 26
Secondary Energy|electr|h2_bio_ccs|M1                             26
Secondary Energy|electr|h2_smr|M1                                 26
                                                                  ..
Emissions|SO2|eth_rc|M1                                           13
Emissions|SO2|flaring_CO2|M1                                      13
Emissions|SO2|foil_hpl|M1                                         13
Emissions|SO2|foil_i|M1                                           13
out|water_supply|upstream_landuse|extract__upstream_landuse|M1    13
Name: count, Length: 3342, dtype: int64


In [30]:
report_df.columns

Index(['Model', 'Scenario', 'Region', 'Variable', 'Unit', '2010', '2015',
       '2020', '2025', '2030', '2035', '2040', '2045', '2050', '2055', '2060'],
      dtype='object')

In [64]:
test = report_df['variable'].value_counts()
# print("Number of unique variables:", report_df['Variable'].nunique())


variable
Secondary Energy|electr|h2_smr_ccs|M1                             26
Secondary Energy|electr|h2_coal|M1                                26
Secondary Energy|electr|h2_bio|M1                                 26
Secondary Energy|electr|h2_bio_ccs|M1                             26
Secondary Energy|electr|h2_smr|M1                                 26
                                                                  ..
Emissions|SO2|eth_rc|M1                                           13
Emissions|SO2|flaring_CO2|M1                                      13
Emissions|SO2|foil_hpl|M1                                         13
Emissions|SO2|foil_i|M1                                           13
out|water_supply|upstream_landuse|extract__upstream_landuse|M1    13
Name: count, Length: 3342, dtype: int64

In [6]:
#from message_ix_models.tools.costs.config import Config
#from message_ix_models.tools.costs.projections import create_cost_projections

In [ ]:
'''# Step 1: Define hydrogen-related technologies
hydrogen_tech_list = ["h2_atr", "h2_atr_ccs"]

# Step 2: Define configuration for cost projections using the 'convergence' method
cfg = Config(
    module="energy",  # Assuming hydrogen falls under energy module
    method="convergence",  # Use convergence method for cost projections
    base_year=2020,  # Base year for the cost projections
    convergence_year=2050,  # Year to which costs should converge
    node="R12",  # Set your region, e.g., R12 (or your specific region code)
    ref_region="R12_NAM"  # Set the reference region
)

# Step 3: Generate base cost projections for all energy technologies
cost_projections = create_cost_projections(cfg)

# Step 4: Extract investment and fixed cost DataFrames
inv_cost_df = cost_projections["inv_cost"]
fix_cost_df = cost_projections["fix_cost"]

# Print columns for debugging
print("Investment Cost DataFrame Columns:")
print(inv_cost_df.columns)

print("\nFixed Cost DataFrame Columns:")
print(fix_cost_df.columns)

# Step 5: Filter cost projections for hydrogen-related technologies
hydrogen_inv_cost = inv_cost_df[inv_cost_df['technology'].isin(hydrogen_tech_list)]

# Debugging: Print the filtered DataFrame to understand its structure
print("\nFiltered Hydrogen Investment Cost DataFrame:")
print(hydrogen_inv_cost.head())

# Check if 'year' column is present
if 'year' not in hydrogen_inv_cost.columns:
    print("No 'year' column found in hydrogen investment costs. Available columns are:", hydrogen_inv_cost.columns)

# Step 6: Apply the learning rate manually to the investment cost
learning_rate = 0.1  # 10% learning rate

# Use 'year' instead of 'year_act' for base year costs
base_year_costs = hydrogen_inv_cost[hydrogen_inv_cost['year'] == 2020].set_index('technology')['inv_cost']

# Apply learning rate decay to future years
for year in hydrogen_inv_cost['year'].unique():
    if year > 2020:
        for tech in hydrogen_tech_list:
            # Calculate the new cost for each technology
            if tech in base_year_costs.index:
                new_cost = base_year_costs[tech] * (1 - learning_rate) ** ((year - 2020) / 5)
                # Update the DataFrame
                hydrogen_inv_cost.loc[
                    (hydrogen_inv_cost['year'] == year) & (hydrogen_inv_cost['technology'] == tech), 'inv_cost'] = new_cost

# Step 7: Extract final investment and fixed costs
inv_cost = hydrogen_inv_cost[['technology', 'year', 'inv_cost']] if 'inv_cost' in hydrogen_inv_cost.columns else None
fix_cost = fix_cost_df[fix_cost_df['technology'].isin(hydrogen_tech_list)] if 'technology' in fix_cost_df.columns else None

# You can now apply these adjusted costs to your MESSAGEix model.

# Print adjusted costs for verification
print("\nAdjusted Investment Costs:")
print(inv_cost)'''

In [5]:
msg.scenario.solve(
					solve_options={"barcrossalg": "2", "lpmethod": "1"})

In [18]:
msg_plt = Plotter_MSG(msg_scenario)
nodes = msg_scenario.set('node')[2:]


In [ ]:
msg_plt.plotter(case="test",nodeloc=nodes,path="output",yr_min=2020, yr_max=2060)